In [5]:
import pygame
import numpy as np
import tkinter as tk
from tkinter import messagebox, simpledialog,filedialog
from pydub.generators import Sine
import random
import time
import math
import os

# 初始化 pygame
pygame.init()
pygame.mixer.init()

# 視窗大小
WIDTH, HEIGHT = 800, 600
screen = pygame.display.set_mode((WIDTH, HEIGHT))
pygame.display.set_caption("可以錄製的兒童鋼琴遊戲")

# 顏色設定
BLACK = (0, 0, 0)
WHITE = (255, 255, 255)
YELLOW=(255,255,0)

# 音符頻率 (Hz)
note_frequencies = {
    pygame.K_a: 261.63,  # Do
    pygame.K_s: 293.66,  # Re
    pygame.K_d: 329.63,  # Mi
    pygame.K_f: 349.23,  # Fa
    pygame.K_g: 392.00,  # Sol
    pygame.K_h: 440.00,  # La
    pygame.K_j: 493.88,  # Ti
    pygame.K_k: 523.25,  # High Do
}

# 錄製相關
recording = False
recorded_notes = []

# 音符生成
DURATION = 1  # 音符持續時間（秒）
VOLUME = 50    # 音量

def create_tone(frequency, duration):
    return Sine(frequency).to_audio_segment(duration=duration * 1000, volume=-20)

def show_cover():
    screen.fill(BLACK)

    # 使用 NotoSansCJKtc-Regular.otf 字體
    title_font = pygame.font.Font("NotoSansCJKtc-Regular.otf", 50)
    button_font = pygame.font.Font("NotoSansCJKtc-Regular.otf", 30)

    title = title_font.render("可以錄製的兒童鋼琴遊戲", True, WHITE)
    start_text = button_font.render("開始錄製", True, WHITE)
    instructions_text = button_font.render("遊戲說明", True, WHITE)
    exit_text = button_font.render("離開", True, WHITE)

    # 設定按鈕位置
    start_rect = start_text.get_rect(center=(WIDTH // 2, 300))
    instructions_rect = instructions_text.get_rect(center=(WIDTH // 2, 400))
    exit_rect = exit_text.get_rect(center=(WIDTH // 2, 500))

    # 繪製標題和按鈕
    screen.blit(title, (WIDTH // 2 - title.get_width() // 2, 100))
    screen.blit(start_text, start_rect)
    screen.blit(instructions_text, instructions_rect)
    screen.blit(exit_text, exit_rect)

    # 在右上角繪製黃色星星
    draw_pentagram(WIDTH - 50, 50, 30)  # 這裡繪製星星

    pygame.display.flip()
    return start_rect, instructions_rect, exit_rect, start_text, instructions_text, exit_text

def draw_pentagram(x, y, size=30):
    # 計算五角星的外圍點
    outer_points = []
    inner_points = []

    for i in range(5):
        outer_angle = math.radians(90 + i * 72)  # 外圍角度
        inner_angle = math.radians(126 + i * 72)  # 內部角度
        outer_points.append((x + size * math.cos(outer_angle), y - size * math.sin(outer_angle)))
        inner_points.append((x + size * 0.5 * math.cos(inner_angle), y - size * 0.5 * math.sin(inner_angle)))

    # 繪製五角星的完整路徑
    star_points = []
    for i in range(5):
        star_points.append(outer_points[i])
        star_points.append(inner_points[i])

    pygame.draw.polygon(screen, YELLOW, star_points)

def is_star_clicked(pos):
    x, y = pos
    distance = math.sqrt((x - (WIDTH - 50))**2 + (y - 50)**2)
    return distance <= 30

def browse_and_play_wav():
    root = tk.Tk()
    root.withdraw()  # 隱藏主視窗

    # 停止背景音樂和其他音效
    pygame.mixer.music.pause()
    pygame.mixer.stop()

    # 瀏覽檔案
    file_path = filedialog.askopenfilename(initialdir=os.getcwd(), title="選擇 WAV 檔案", filetypes=[("WAV Files", "*.wav")])
    if file_path:
        if file_path.endswith(".wav"):
            try:
                sound = pygame.mixer.Sound(file_path)
                sound.play()
                messagebox.showinfo("播放成功", f"正在播放: {os.path.basename(file_path)}")
            except Exception as e:
                messagebox.showerror("播放失敗", f"無法播放檔案: {e}")
        else:
            messagebox.showwarning("檔案格式錯誤", "請選擇 WAV 格式的檔案")
    else:
        # 如果未選擇檔案，繼續播放背景音樂
        pygame.mixer.music.unpause()

    root.destroy()



def highlight_button(button_text, button_rect, highlighted=True):
 
    screen.fill((0, 0, 0), button_rect)  # 用黑色清空原有的區域

    # 如果高亮顯示，將背景改為白色，文字為黑色
    if highlighted:
        button_color = (255, 255, 255)  
        text_color = (0, 0, 0)  
    else:
        button_color = (0, 0, 0)  
        text_color = (255, 255, 255)  

    # 畫出按鈕背景
    pygame.draw.rect(screen, button_color, button_rect)

    # 用設定的顏色渲染文字並放置
    button_font = pygame.font.Font("NotoSansCJKtc-Regular.otf", 30)
    text = button_font.render(button_text, True, text_color)
    screen.blit(text, (button_rect.x + (button_rect.width - text.get_width()) // 2, 
                       button_rect.y + (button_rect.height - text.get_height()) // 2))

    pygame.display.update()

def save_recording():
    if recorded_notes:
        root = tk.Tk()
        root.withdraw()  

        root.geometry("00x150") 

        file_name = simpledialog.askstring("檔案名稱", "請輸入檔案名稱（不包含副檔名）:", parent=root)
        
        if file_name:
            file_name_with_extension = file_name + ".wav"
            
            combined = sum(recorded_notes)
            combined.export(file_name_with_extension, format="wav")
            messagebox.showinfo("錄製完成", f"錄製的音檔已儲存為 {file_name_with_extension}")
        else:
            messagebox.showwarning("警告", "您未輸入檔案名稱，檔案未儲存。")
        
        root.destroy()

def create_silence(duration):
    """Create a silent audio segment for a given duration in seconds."""
    return AudioSegment.silent(duration=duration * 1000)  # PyDub uses milliseconds

last_key_time = 0 

def game_loop():
    global recording, recorded_notes

    running = True
    game_started = False
    show_cover_screen = True
    button_highlighted = None

    pygame.mixer.music.load("cover_music.mp3")
    pygame.mixer.music.play(-1, 0.0)


    # 音符名稱字體
    font = pygame.font.Font("NotoSansCJKtc-Regular.otf", 80)

    # 儲存顯示的音符名稱、顏色與位置
    displayed_notes = []

    while running:
        if show_cover_screen:
            start_rect, instructions_rect, exit_rect, start_text, instructions_text, exit_text = show_cover()  # 顯示封面畫面
        else:
            screen.fill(BLACK)  # 清空畫面

            # 顯示目前的音符名稱並隨著時間亂移動
            for note in displayed_notes:
                note_text = font.render(note['name'], True, note['color'])
                screen.blit(note_text, (note['x'], note['y']))

        pygame.display.flip()

        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                running = False
            if event.type == pygame.MOUSEBUTTONDOWN:
                pos = pygame.mouse.get_pos()
                if is_star_clicked(pos):  # 檢測是否點擊了星星
                    browse_and_play_wav()

            if not game_started and event.type == pygame.MOUSEBUTTONDOWN:
                pos = pygame.mouse.get_pos()

                if start_rect.collidepoint(pos):
                    button_highlighted = "start"
                    highlight_button("開始錄製", start_rect)
                    pygame.time.wait(300)
                    highlight_button("開始錄製", start_rect, highlighted=False)

                    game_started = True
                    recording = True
                    displayed_notes = []
                    recorded_notes = []
                    pygame.mixer.music.stop()
                    pygame.display.set_caption("錄製中...")
                    show_cover_screen = False
                    screen.fill(BLACK)

                elif instructions_rect.collidepoint(pos):
                    button_highlighted = "instructions"
                    highlight_button("遊戲說明", instructions_rect)
                    pygame.time.wait(300)
                    highlight_button("遊戲說明", instructions_rect, highlighted=False)

                    root = tk.Tk()
                    root.withdraw()
                    messagebox.showinfo("遊戲說明", "按鍵 A、S、D... 可演奏音符。\n按下 Q 結束錄製並保存音檔。")
                    root.destroy()

                elif exit_rect.collidepoint(pos):
                    button_highlighted = "exit"
                    highlight_button("離開", exit_rect)
                    pygame.time.wait(300)
                    highlight_button("離開", exit_rect, highlighted=False)

                    running = False

            if game_started:
                if recording:
                    if event.type == pygame.KEYDOWN:
                        if event.key == pygame.K_q and recording:
                            recording = False
                            pygame.display.set_caption("錄製結束")  
                            save_recording()  
                            pygame.display.set_caption("可以錄製的兒童鋼琴遊戲")  
                            game_started = False
                            pygame.mixer.music.play(-1, 0.0)  
                            show_cover_screen = True
                        elif event.key in note_frequencies:
                            frequency = note_frequencies[event.key]
                            tone = create_tone(frequency, DURATION)
                            recorded_notes.append(tone)

                            samples = np.array(tone.get_array_of_samples()).astype(np.int16)
                            samples_stereo = np.column_stack((samples, samples))  

                            sound = pygame.sndarray.make_sound(samples_stereo)
                            sound.play()

                            # 音符名稱對應
                            note_names = {
                                pygame.K_a: "Do",
                                pygame.K_s: "Re",
                                pygame.K_d: "Mi",
                                pygame.K_f: "Fa",
                                pygame.K_g: "Sol",
                                pygame.K_h: "La",
                                pygame.K_j: "Ti",
                                pygame.K_k: "Do",
                            }

                            # 定義彩虹顏色
                            rainbow_colors = {
                                "Do": (255, 0, 0),      # 紅色
                                "Re": (255, 127, 0),    # 橙色
                                "Mi": (255, 255, 0),    # 黃色
                                "Fa": (0, 255, 0),      # 綠色
                                "Sol": (0, 0, 255),     # 藍色
                                "La": (75, 0, 130),     # 靛色
                                "Ti": (148, 0, 211),    # 紫色
                            }

                            displayed_note = note_names[event.key]

                            displayed_note = note_names[event.key]
                            note_color = rainbow_colors[displayed_note]

                            # 隨機角度與速度
                            angle = random.uniform(0, 2 * math.pi)  # 隨機角度 (0 - 2π)
                            speed = random.uniform(0.5, 1)  # 隨機速度 (1 到 3 像素)
                            dx = speed * math.cos(angle)
                            dy = speed * math.sin(angle)

                            # 隨機位置
                            x_pos = random.randint(0, WIDTH - 100)
                            y_pos = random.randint(0, HEIGHT - 100)

                            displayed_notes.append({
                                'name': displayed_note,
                                'color': note_color,
                                'x': x_pos,
                                'y': y_pos,
                                'dx': dx,
                                'dy': dy
                            })

        # 更新音符位置，並檢查是否碰到邊緣反彈
        for note in displayed_notes:
            note['x'] += note['dx']
            note['y'] += note['dy']

            # 如果碰到邊緣反彈
            if note['x'] <= 0 or note['x'] >= WIDTH - 100:
                note['dx'] = -note['dx']  # 改變水平方向

            if note['y'] <= 0 or note['y'] >= HEIGHT - 100:
                note['dy'] = -note['dy']  # 改變垂直方向

        pygame.display.update()

    pygame.quit()

game_loop()
